<a href="https://colab.research.google.com/github/eehujnihs21-stack/app0320/blob/main/2555037%EC%8B%A0%EC%A3%BC%ED%9D%AC_%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastapi uvicorn httpx beautifulsoup4 gradio pyngrok nest_asyncio pandas deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.0 MB/s eta 0:00:00


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⚠️ NGROK_TOKEN 확인 필수!
NGROK_TOKEN = "3DNLGoFzIkfzR8yfEM7o5Zq5pkY_3VWPMLUTgAVrpNsdRZUt4"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import sqlite3, re, threading, time, random
import pandas as pd
import httpx
import uvicorn
import nest_asyncio
from collections import Counter
from bs4 import BeautifulSoup
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from gradio.routes import mount_gradio_app
from deep_translator import GoogleTranslator
import gradio as gr
from pyngrok import ngrok

nest_asyncio.apply()

# ── 1. DB 및 초기화 (처음 코드 유지) ──────────
DB_PATH = "quotes.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DROP TABLE IF EXISTS quotes")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS quotes (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            text       TEXT NOT NULL,
            author     TEXT NOT NULL,
            tags       TEXT,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit(); conn.close()

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

init_db()

# ── 2. 로직 함수들 (처음 코드 유지) ───────────
def crawl_quotes_logic():
    quotes = []
    try:
        for page in range(1, 3):
            r = httpx.get(f"http://quotes.toscrape.com/page/{page}/", timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")
            for q in soup.select(".quote"):
                quotes.append({
                    "text": q.select_one(".text").get_text(strip=True),
                    "author": q.select_one(".author").get_text(strip=True),
                    "tags": ", ".join(t.get_text() for t in q.select(".tag")),
                })
        conn = get_conn()
        conn.execute("DELETE FROM quotes")
        for q in quotes[:20]:
            conn.execute("INSERT INTO quotes(text,author,tags) VALUES(?,?,?)", (q["text"], q["author"], q["tags"]))
        conn.commit(); conn.close()
        return f"✅ {len(quotes[:20])}개 수집 완료!"
    except Exception as e:
        return f"❌ 크롤링 실패: {e}"

def classify_quote(text):
    text = text.lower()
    if any(k in text for k in ["love","heart","kiss"]): return "❤️ 사랑"
    elif any(k in text for k in ["success","dream","goal","win"]): return "🔥 성공"
    elif any(k in text for k in ["life","living","human"]): return "🌱 인생"
    elif any(k in text for k in ["pain","hard","try","work"]): return "💪 동기부여"
    return "✨ 기타"

# ── 3. FastAPI 설정 (처음 보낸 API 구조 100% 유지) ──
app = FastAPI(
    title="격언 관리 시스템 API",
    swagger_ui_parameters={"docExpansion": "full"}
)

class QuoteIn(BaseModel):
    text: str; author: str; tags: str = ""

@app.get("/quotes", summary="전체 격언 조회")
def api_read_all():
    conn = get_conn()
    rows = conn.execute("SELECT * FROM quotes").fetchall()
    conn.close()
    return [dict(r) for r in rows]

@app.post("/quotes", summary="격언 신규 등록")
def api_create(q: QuoteIn):
    conn = get_conn()
    cur = conn.execute("INSERT INTO quotes(text,author,tags) VALUES(?,?,?)", (q.text, q.author, q.tags))
    conn.commit(); conn.close()
    return {"id": cur.lastrowid, "status": "success"}

@app.put("/quotes/{qid}", summary="격언 정보 수정")
def api_update(qid: int, q: QuoteIn):
    conn = get_conn()
    cur = conn.execute("UPDATE quotes SET text=?, author=?, tags=? WHERE id=?", (q.text, q.author, q.tags, qid))
    if cur.rowcount == 0:
        conn.close()
        raise HTTPException(status_code=404, detail="ID를 찾을 수 없습니다.")
    conn.commit(); conn.close()
    return {"status": "updated"}

@app.delete("/quotes/{qid}", summary="격언 삭제")
def api_delete(qid: int):
    conn = get_conn()
    conn.execute("DELETE FROM quotes WHERE id=?", (qid,))
    conn.commit(); conn.close()
    return {"status": "success"}

# ── 4. Gradio UI 연동 함수 (처음 코드 유지) ─────
def ui_get_all():
    conn = get_conn()
    rows = conn.execute("SELECT id, author, text, tags FROM quotes ORDER BY id ASC").fetchall()
    conn.close()
    return pd.DataFrame([dict(r) for r in rows]) if rows else pd.DataFrame(columns=["id","author","text","tags"])

def ui_filtered_trans(category_emoji):
    conn = get_conn(); rows = conn.execute("SELECT author, text FROM quotes").fetchall(); conn.close()
    translator = GoogleTranslator(source="en", target="ko")
    res = []
    for r in rows:
        cat = classify_quote(r["text"])
        if category_emoji in cat:
            try: ko = translator.translate(r["text"])
            except: ko = "번역 실패"
            res.append({"감성": cat, "저자": r["author"], "원문": r["text"], "해석": ko})
    return pd.DataFrame(res) if res else pd.DataFrame([{"안내": "데이터가 없습니다."}])

def ui_random_quote():
    conn = get_conn()
    row = conn.execute("SELECT text, author FROM quotes ORDER BY RANDOM() LIMIT 1").fetchone()
    conn.close()
    if not row: return "❌ 크롤링을 먼저 진행해주세요!"
    try: ko = GoogleTranslator(source="en", target="ko").translate(row["text"])
    except: ko = "번역 실패"
    msgs = ["🌞 오늘은 당신의 날입니다!", "✨ 당신의 노력을 믿으세요!", "🍀 행운이 함께할 거예요!", "🔥 포기하지 마세요!"]
    return f"📖 원문: {row['text']}\n\n🇰🇷 해석: {ko}\n\n✍️ 저자: {row['author']}\n\n━━━━━━━━━━━━━━━\n{random.choice(msgs)}"

def ui_word_stats():
    conn = get_conn(); rows = conn.execute("SELECT text FROM quotes").fetchall(); conn.close()
    if not rows: return pd.DataFrame(columns=["단어", "빈도수"])
    stop = {"the","a","an","of","in","to","and","is","it","that","for","with","you","i","be","on","are","as","we","this"}
    words = [w for r in rows for w in re.findall(r"[a-z]+", r["text"].lower()) if w not in stop and len(w) > 3]
    top = Counter(words).most_common(15)
    return pd.DataFrame(top, columns=["단어", "빈도수"])

def ui_author_stats():
    conn = get_conn(); total_row = conn.execute("SELECT COUNT(*) FROM quotes").fetchone(); total_count = total_row[0] if total_row else 0
    rows = conn.execute("SELECT author, COUNT(*) as count FROM quotes GROUP BY author ORDER BY count DESC LIMIT 10").fetchall(); conn.close()
    if not rows or total_count == 0: return {}
    return {r["author"]: r["count"] / total_count for r in rows}

# ── 5. Gradio UI 구성 (요청하신 순서로 배치) ─────
with gr.Blocks(title="격언 종합 대시보드") as ui:
    gr.Markdown("# 📚 격언 관리 및 분석 시스템")

    # 1. 데이터 수집/목록
    with gr.Tab("🕷️ 데이터 수집/목록"):
        with gr.Row():
            btn_crawl = gr.Button("웹 크롤링 실행", variant="primary")
            btn_refresh = gr.Button("목록 새로고침")
        txt_status = gr.Textbox(label="상태")
        tbl_main = gr.Dataframe(label="전체 격언 목록")
        btn_crawl.click(crawl_quotes_logic, outputs=txt_status)
        btn_refresh.click(ui_get_all, outputs=tbl_main)

    # 2. 감성 필터링 & 해석
    with gr.Tab("🧠 감성 필터링 & 해석"):
        with gr.Row():
            b1 = gr.Button("❤️ 사랑"); b2 = gr.Button("🔥 성공"); b3 = gr.Button("🌱 인생"); b4 = gr.Button("💪 동기부여")
        tbl_filter = gr.Dataframe(wrap=True)
        b1.click(lambda: ui_filtered_trans("❤️"), outputs=tbl_filter)
        b2.click(lambda: ui_filtered_trans("🔥"), outputs=tbl_filter)
        b3.click(lambda: ui_filtered_trans("🌱"), outputs=tbl_filter)
        b4.click(lambda: ui_filtered_trans("💪"), outputs=tbl_filter)

    # 3. 오늘의 격언
    with gr.Tab("🍀 오늘의 격언"):
        out_random = gr.Textbox(label="격언 카드", lines=10, interactive=False)
        btn_random = gr.Button("새로운 격언 뽑기", variant="primary")
        btn_random.click(ui_random_quote, outputs=out_random)

    # 4. 데이터 통계
    with gr.Tab("📊 데이터 통계"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🔤 단어 빈도 분석")
                btn_stats = gr.Button("단어 분석 실행")
                plt_stats = gr.BarPlot(x="단어", y="빈도수", vertical=False)
                btn_stats.click(ui_word_stats, outputs=plt_stats)
            with gr.Column():
                gr.Markdown("### ✍️ 작가별 점유율")
                btn_author_stats = gr.Button("작가 분석 실행")
                plt_author_stats = gr.Label(label="비율", num_top_classes=10)
                btn_author_stats.click(ui_author_stats, outputs=plt_author_stats)

    # 5. 추가/삭제/수정
    with gr.Tab("✏️ 추가/삭제/수정"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🛠️ 데이터 추가 및 수정")
                in_qid = gr.Textbox(label="수정용 ID (추가는 빈칸)")
                in_txt = gr.Textbox(label="내용"); in_aut = gr.Textbox(label="저자"); in_tag = gr.Textbox(label="태그")
                with gr.Row():
                    btn_add = gr.Button("신규 추가", variant="primary")
                    btn_upd = gr.Button("정보 수정", variant="secondary")
                out_crud = gr.Textbox(label="결과")
                btn_add.click(lambda t,a,g: api_create(QuoteIn(text=t,author=a,tags=g)), [in_txt, in_aut, in_tag], out_crud)
                btn_upd.click(lambda i,t,a,g: api_update(int(i), QuoteIn(text=t,author=a,tags=g)), [in_qid, in_txt, in_aut, in_tag], out_crud)
            with gr.Column():
                gr.Markdown("### 🗑️ 데이터 삭제")
                in_del = gr.Textbox(label="삭제할 ID")
                btn_del = gr.Button("삭제 실행", variant="stop")
                out_del = gr.Textbox(label="결과")
                btn_del.click(lambda q: api_delete(int(q)), in_del, out_del)

# ── 6. 서버 실행 ─────────────────────────────
app = mount_gradio_app(app, ui, path="/ui")

def run_srv():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run_srv, daemon=True).start()
time.sleep(3)

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
url = ngrok.connect(8000).public_url

print(f"\n✅ 서버 가동 완료!")
print(f"🌍 사용자 UI: {url}/ui")
print(f"📑 API 문서: {url}/docs  <-- 이 주소로 들어가면 API가 뜹니다!")

new /ui


INFO:     Started server process [9829]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



✅ 서버 가동 완료!
🌍 사용자 UI: https://mop-reclusive-approval.ngrok-free.dev/ui
📑 API 문서: https://mop-reclusive-approval.ngrok-free.dev/docs  <-- 이 주소로 들어가면 API가 뜹니다!
